# Inpactor3-SegNT · Pipeline completo PanTEon cross-kingdom

Entrena SegNT sobre PanTEon completo (plantas + hongos + animales) usando
el dataset común y valida sobre 4 genomas de referencia.

**Antes de correr**: `Runtime → Change runtime type → T4 GPU`.

**Duración total estimada**: ~6-8 h en GPU T4 (Colab gratuito).

**IMPORTANTE**: No cierres la pestaña durante el entrenamiento. Salva
checkpoint a Drive antes de cerrar Colab.

## 1. Verificar GPU

In [ ]:
!nvidia-smi

## 2. Clonar repo e instalar

In [ ]:
%cd /content
!rm -rf Inpactor3-SegNT
!git clone https://github.com/Inpactor3/Inpactor3-SegNT.git
%cd Inpactor3-SegNT
!pip install -q -e .

## 3. Descargar PanTEon (~1 GB, 2-3 min con red Colab)

In [ ]:
!mkdir -p data/raw
!curl -Lk -o data/raw/PanTEon_Database_v1.6.2.fasta \
  "https://zenodo.org/api/records/21372179/files/PanTEon_Database_v1.6.2.fasta/content"
!curl -Lk -o data/raw/PanTEon_Database_metadata_v1.6.2.csv \
  "https://zenodo.org/api/records/21372179/files/PanTEon_Database_metadata_v1.6.2.csv/content"
!ls -lh data/raw/

## 4. Descargar genomas de fondo (~5 GB, 5-10 min)

In [ ]:
!bash scripts/download_backgrounds.sh

## 5. Construir dataset común (~20-30 min)

In [ ]:
!python scripts/build_teon_dataset.py \
    --panteon data/raw/PanTEon_Database_v1.6.2.fasta \
    --metadata data/raw/PanTEon_Database_metadata_v1.6.2.csv \
    --genomes-dir data/common_dataset/genomes \
    --out-dir data/common_dataset \
    --n-positives 30000 \
    --n-negatives 30000
!ls -lh data/common_dataset/

## 6. Entrenar SegNT (encoder congelado, ~3-4 h GPU T4)

In [ ]:
!python -m inpactor3_segnt.train_teon --config configs/nt_50m_teon.yaml

## 7. Descargar 4 genomas de validación (~30 min, ~4 GB)

In [ ]:
!mkdir -p data/validation_genomes
%cd data/validation_genomes

# Arabidopsis (~120 MB)
!for CHR in 1 2 3 4 5; do \
  curl -Lk -o "chr${CHR}.fa.gz" \
    "https://ftp.ensemblgenomes.org/pub/plants/release-58/fasta/arabidopsis_thaliana/dna/Arabidopsis_thaliana.TAIR10.dna_sm.chromosome.${CHR}.fa.gz"; \
  gunzip "chr${CHR}.fa.gz"; \
done && cat chr*.fa > Arabidopsis_thaliana.fasta && rm chr*.fa

# Oryza sativa (~370 MB)
!curl -Lk -o Oryza_sativa.fa.gz \
  "https://ftp.ensemblgenomes.org/pub/plants/release-58/fasta/oryza_sativa/dna/Oryza_sativa.IRGSP-1.0.dna_sm.toplevel.fa.gz"
!gunzip Oryza_sativa.fa.gz && mv Oryza_sativa.fa Oryza_sativa.fasta

# Ananas comosus (~500 MB)
!curl -Lk -o Ananas_comosus.fa.gz \
  "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/001/540/865/GCF_001540865.1_ASM154086v1/GCF_001540865.1_ASM154086v1_genomic.fna.gz"
!gunzip Ananas_comosus.fa.gz && mv Ananas_comosus.fa Ananas_comosus.fasta

# Zea mays (usar el que ya bajamos)
!cp ../common_dataset/genomes/Zea_mays.fasta Zea_mays.fasta

%cd /content/Inpactor3-SegNT
!ls -lh data/validation_genomes/

## 8. Inferencia sobre los 4 genomas (~2 h con GPU)

In [ ]:
!mkdir -p results/segnt

for genome in ['Arabidopsis_thaliana', 'Oryza_sativa', 'Ananas_comosus', 'Zea_mays']:
    print(f'\n=== {genome} ===')
    !python -m inpactor3_segnt.predict \
        --checkpoint models/best_teon.pt \
        --genome data/validation_genomes/{genome}.fasta \
        --out results/segnt/{genome}_predictions.tab \
        --threshold 0.5

## 9. Reporte final (% LTR-RT vs esperado biológico)

In [ ]:
!python scripts/validate_on_genomes.py \
    --predictions-dir results/segnt \
    --genomes-dir data/validation_genomes \
    --model-name "Inpactor 3-SegNT (PanTEon cross-kingdom)" \
    --report results/validation_segnt.md

!cat results/validation_segnt.md

## 10. Guardar TODO en Google Drive (CRÍTICO antes de cerrar)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/Inpactor3_teon
!cp models/best_teon.pt /content/drive/MyDrive/Inpactor3_teon/
!cp -r results/ /content/drive/MyDrive/Inpactor3_teon/
!cp -r logs/ /content/drive/MyDrive/Inpactor3_teon/
!ls -lh /content/drive/MyDrive/Inpactor3_teon/